# [데모 A] 슬라이드 흐름형 — Tool Calling

> **교안 형식 후보 A.** 슬라이드 한 장 → 해설 → 바로 실행 → 결과 확인을 반복한다. 진행 순서가 PDF와 같아서 강사가 따라가기 쉽다.

| | |
|---|---|
| 커리큘럼 | 1.3 LangChain 프레임워크 깊이 학습 — Tool Calling |
| 슬라이드 | `docs/raw/Chapter01/day01/1-3-langchain-framework-deep-dive.pdf` p9~11 |
| 실습 원본 | `src/raw/Chapter01/02-langchain/02_tool_calling.py` |
| 실행 조건 | LLM을 호출한다(`.env`의 `LLM_PROVIDER`). 날씨·계산 도구는 파일 안의 가짜 데이터 함수다. |

## 왜 배우나 · 언제 쓰나

- LLM은 글을 만든다. 날씨 조회나 정확한 계산 같은 **행동**은 직접 하지 못한다.
- Tool Calling은 모델이 "어떤 도구를 어떤 인자로 부를지"를 고르고, **실제 실행은 우리 코드가 하는** 방식이다 (슬라이드 p9·p10).
- 이 선택 → 실행 → 관찰을 반복하면 Agent의 기본 실행 흐름이 된다 (p11). 원본 코드 끝의 주석대로, 뒤에서 배우는 LangGraph `ToolNode`가 이 반복을 자동화한다.

In [ ]:
from assets.widgets import show_checklist, show_slides, show_trace

## 1. 도구 스키마 — 모델은 무엇을 보고 고르나

In [ ]:
show_slides("1-3", [9])

슬라이드의 요점은 "좋은 도구 설명은 기능뿐 아니라 **언제 써야 하는지**를 말해야 한다"이다.
아래 코드에서 `@tool`을 붙인 함수의 **docstring은 도구 설명**이 되고, **타입 힌트는 인자 형식**이 된다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv()


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    fake_data = {
        '서울': '맑음, 기온 22°C',
        '부산': '흐림, 기온 24°C',
        '제주': '비, 기온 20°C',
    }
    return fake_data.get(city, f'{city}의 날씨 데이터를 찾을 수 없습니다.')


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    try:
        result = eval(expression, {'__builtins__': {}})  # noqa: S307
        return str(result)
    except Exception as e:
        return f'계산 오류: {e}'


tools = [get_weather, calculate]

모델에게 실제로 전달되는 스키마를 출력해 본다. **볼 지점:** `description`이 docstring과 같고, `parameters`에 `city: string`이 있다.

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(get_weather), ensure_ascii=False, indent=2))

## 2. 모델은 실행하지 않는다 — 호출 의도만 만든다

In [ ]:
show_slides("1-3", [10])

In [ ]:
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)   # 호출할 때마다 도구 스키마를 함께 보낸다

response = llm_with_tools.invoke([HumanMessage(content='서울 날씨 알려줘')])
print('content   :', repr(response.content))
print('tool_calls:', response.tool_calls)

**볼 지점:** 응답에는 도구 이름과 인자(`tool_calls`)가 들어 있다. `get_weather` 함수는 **아직 한 번도 실행되지 않았다.**
실행은 다음 단계에서 우리 코드가 한다.

## 3. 반복 루프 — 실행하고, 결과를 돌려주고, 다시 판단한다

In [ ]:
show_slides("1-3", [11])

원본의 수동 루프 그대로다. 단, 흐름을 그림으로 보려고 **마지막 줄만 바꿔 `messages`도 함께 돌려준다** (원본은 답만 돌려준다).

In [ ]:
tool_map = {tool.name: tool for tool in tools}   # 도구 이름 → 실행할 함수


def run_with_tools(question: str):
    messages = [HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content, messages   # 원본: return response.content

        for call in response.tool_calls:
            tool_fn = tool_map[call['name']]
            result = tool_fn.invoke(call['args'])
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

In [ ]:
answer, messages = run_with_tools('서울과 부산 날씨를 비교해줘')
show_trace(messages)

**볼 지점**
- 슬라이드 p11의 1~4단계(판단 → 도구 선택 → 결과 관찰 → 반복 또는 답변)를 카드에서 찾는다.
- 모델에 따라 두 도시를 한 번에 부르기도 하고 차례로 부르기도 한다. 어느 쪽이든 `tool_call_id`로 요청과 결과의 짝이 맞는다. 도구 호출 카드를 클릭해 확인한다.

## 4. CodeMate 실습 — 도구 하나 추가하고 리뷰하기

먼저 **지금 상태로** 실행해 본다. 환율 도구가 없을 때 모델이 어떻게 답하는지 기록해 둔다.

In [ ]:
answer, messages = run_with_tools('100달러는 원화로 얼마야?')
show_trace(messages)

**볼 지점:** 환율 도구가 없으면 모델이 환율을 **스스로 가정해** `calculate`에 넣을 수 있다. 카드에서 그 숫자가 어디서 왔는지 확인한다. 도구가 아니라 모델이 만든 값이라면, 그것이 도구를 추가해야 하는 이유다.

아래 지시문을 CodeMate 채팅에 붙여 넣는다.

> 이 노트북 1번의 도구 정의 셀에 `get_exchange_rate(currency: str)` 도구를 추가해 줘. 가짜 데이터로 USD, JPY, EUR의 원화 환율을 돌려주고, 없는 통화는 안내 문구를 돌려줘. docstring에는 무엇을 하는지와 **언제 이 도구를 써야 하는지**를 적어 줘. `tools` 목록에도 넣어 줘.

CodeMate가 고친 셀부터 차례로 다시 실행하고, 바로 위 셀로 같은 질문을 다시 해 본다. 결과를 아래 기준으로 리뷰한다. 기준에 못 미치면 무엇이 부족한지 적어 다시 지시한다.

In [ ]:
show_checklist([
    'docstring이 "무엇을 하는지"와 "언제 쓰는지"를 말한다 (슬라이드 p9)',
    '스키마 출력(1번)을 다시 해 보면 parameters에 currency가 보인다',
    'tools 목록에 추가했다 — tool_map은 목록에서 자동으로 만들어지므로 따로 고치지 않았다',
    '메시지 흐름에서 get_exchange_rate가 호출되고, 필요하면 calculate가 이어서 쓰였다',
    '없는 통화를 물으면 오류 대신 안내 문구가 돌아온다',
])